# Carteira Polo Parnaiba - Modelo de previsao de churn

**Pergunta:** um cliente ativo no fechamento do mes vai zerar o TPV no mes seguinte?

**Uso:** todo mes, entregar ao time a lista dos **120 clientes** de maior risco (ate 200 por telefone).

Celulas 1 a 7 preparam a base; 8 a 12 comparam abordagens, explicam os clusters e geram a lista;
13 confere a lista contra o mes realizado; 14 reconstroi as listas mes a mes (backtest).
A preparacao parte da mesma base tratada do dashboard (`base_dashboard.ipynb`, executada por
dentro) e faz tres coisas:

1. **Define o alvo sem vazamento.** O alvo e o churn do mes *seguinte*, observado a partir dos
   clientes ativos no mes corrente. O `flag_churn` do proprio mes nao serve: nele o TPV ja e zero.
2. **Reduz as variaveis**, com o motivo de cada exclusao registrado. Duas colunas saem por
   vazamento comprovado: `dias_sem_transacionar` e `Data_ultima_transacao`.
3. **Resume o TPV diario.** As 31 colunas `tpv_d*` viram 7 medidas de frequencia, recencia,
   buracos, desaceleracao, volatilidade e concentracao; as 2 redundantes entre si saem e ficam 5.

Saida: `df_modelo` em memoria e `data/processed/base_modelo.parquet` (fora do Git).

In [1]:
# =============================================================================
# 1. CARGA DA BASE TRATADA
# =============================================================================
# O modelo parte da MESMA base do dashboard. As celulas `consolida` e `features`
# de base_dashboard.ipynb sao executadas por dentro (src/base_tratada.py), entao
# consolidacao por documento, validacao do PIX, flags de fluxo e Rota_atual sao
# exatamente as do dashboard. Nenhuma regra de tratamento e reimplementada aqui.

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

_RAIZ = next(pasta for pasta in [Path.cwd(), *Path.cwd().parents]
             if (pasta / "data").is_dir() and (pasta / "notebooks").is_dir())
if str(_RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(_RAIZ / "src"))
import caminhos
from base_tratada import carrega_base_tratada

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

_inicio = time.perf_counter()
df = carrega_base_tratada()
print(f"Base tratada: {df.shape[0]:,} linhas x {df.shape[1]} colunas "
      f"({time.perf_counter() - _inicio:,.1f}s)")
print(f"Periodo: {df['periodo'].min()} a {df['periodo'].max()} | "
      f"documentos: {df['Documento'].nunique():,}")

Base tratada: 176,514 linhas x 107 colunas (7.8s)
Periodo: 2024-01 a 2026-07 | documentos: 7,076


## 2. Alvo e populacao

In [2]:
# =============================================================================
# 2. ALVO E POPULACAO
# =============================================================================
# Pergunta do modelo: um cliente ATIVO no fechamento do mes t vai zerar o TPV
# Total no mes t+1?
#
#   alvo_churn_m1 = 1  ->  transacionou em t e ficou zerado em t+1
#
# Por que prever t+1 e nao usar o flag_churn do proprio mes: no mes do churn o TPV
# ja e zero, e toda variavel transacional daquele mes entregaria a resposta. O
# modelo precisa enxergar o cliente ANTES da saida, quando o comportamento ainda
# esta so irregular.
#
# Populacao: transacionou == 1 em t, com 3 meses de historico (abr/2024 em diante).
# Dois recortes:
#   - documento que some da base em t+1 (mudanca de escopo, README 8.4): nao ha
#     como saber se parou de transacionar ou mudou de carteira -> alvo indefinido,
#     linha descartada
#   - ultimo mes da base: nao existe t+1 -> conjunto de ESCORAGEM, os clientes
#     ativos hoje cujo risco o modelo vai estimar

MESES = pd.period_range(df["periodo"].min(), df["periodo"].max(), freq="M")
ULTIMO_MES = MESES[-1]

_painel = (df.pivot_table(index="Documento", columns="periodo", values="tpv_total",
                          aggfunc="sum")
             .reindex(columns=MESES))
_presente = _painel.notna()
_painel = _painel.fillna(0.0)

_futuro = pd.concat(
    [_painel.shift(-1, axis=1).stack().rename("tpv_total_prox"),
     _presente.shift(-1, axis=1, fill_value=False).stack().rename("presente_prox")],
    axis=1,
).rename_axis(["Documento", "periodo"])

base = (df[(df["transacionou"] == 1) & (df["janela_hist_completa"] == 1)]
        .merge(_futuro, how="left", left_on=["Documento", "periodo"], right_index=True))

base["conjunto"] = np.where(base["periodo"] == ULTIMO_MES, "escoragem", "treino")
_treino = base["conjunto"] == "treino"
_zerou = base["tpv_total_prox"] <= 0

_indefinido = _treino & _zerou & ~base["presente_prox"].astype(bool)
print(f"Descartados por alvo indefinido (sumiram da base em t+1): {int(_indefinido.sum())}")
base = base[~_indefinido].copy()

base["alvo_churn_m1"] = (base["tpv_total_prox"] <= 0).astype("Int8")
base.loc[base["conjunto"] == "escoragem", "alvo_churn_m1"] = pd.NA

# Conferencia nos dois sentidos: todo alvo = 1 em t e um flag_churn em t+1, e todo
# flag_churn de mai/2024 em diante (primeiro mes com t dentro da populacao) vira alvo
_churn_seguinte = (df.loc[df["flag_churn"] == 1, ["Documento", "periodo"]]
                     .assign(periodo=lambda x: x["periodo"] - 1))
_confere = base.loc[base["alvo_churn_m1"] == 1, ["Documento", "periodo"]].merge(
    _churn_seguinte, how="left", indicator=True)
_esperado = int(((df["flag_churn"] == 1) & (df["periodo"] >= MESES[0] + 4)).sum())
print(f"Alvos que batem com o flag_churn do mes seguinte: "
      f"{(_confere['_merge'] == 'both').mean():.2%} de {len(_confere):,}")
print(f"flag_churn de {MESES[0] + 4} em diante: {_esperado:,} "
      f"-> {'confere' if _esperado == len(_confere) else 'DIVERGE'}")

_t = base[base["conjunto"] == "treino"]
print(f"\nTreino: {len(_t):,} linhas ({_t['periodo'].min()} a {_t['periodo'].max()}), "
      f"{int(_t['alvo_churn_m1'].sum()):,} churns, "
      f"taxa {_t['alvo_churn_m1'].astype(float).mean():.2%}")
print(f"Escoragem: {int((base['conjunto'] == 'escoragem').sum()):,} "
      f"clientes ativos em {ULTIMO_MES}")
print("\nTaxa de churn no mes seguinte, por ano de t:")
print(_t.groupby(_t["periodo"].dt.year)["alvo_churn_m1"]
        .agg(linhas="size", churns="sum",
             taxa_pct=lambda s: round(s.astype(float).mean() * 100, 2))
        .to_string())

Descartados por alvo indefinido (sumiram da base em t+1): 36
Alvos que batem com o flag_churn do mes seguinte: 100.00% de 2,265
flag_churn de 2024-05 em diante: 2,265 -> confere



Treino: 40,068 linhas (2024-04 a 2026-06), 2,265 churns, taxa 5.65%
Escoragem: 1,559 clientes ativos em 2026-07

Taxa de churn no mes seguinte, por ano de t:
         linhas  churns  taxa_pct
periodo                          
2024      13071     678      5.19
2025      18035    1026      5.69
2026       8962     561      6.26


## 3. Reducao de variaveis

In [3]:
# =============================================================================
# 3. REDUCAO DE VARIAVEIS: O QUE SAI E POR QUE
# =============================================================================
# Cada coluna da base tratada tem destino explicito: vira chave, entra como
# feature ou sai com motivo registrado. A celula 6 recusa montar o dataframe se
# alguma coluna ficar sem destino, para nada entrar no modelo por omissao.
#
# As colunas marcadas como "substituida" ainda sao lidas nas celulas 4 e 5 para
# derivar features mais compactas; so saem no recorte final.
#
# Para devolver uma variavel ao modelo: tire-a daqui e inclua em FEATURES_BASE
# (celula 6).

_DIARIO = [f"tpv_d{d}" for d in range(1, 32)]

MOTIVOS = {
    "sugestao": "Sugestao do usuario: dependente de outra variavel ou fora do escopo",
    "vazamento": "VAZAMENTO: carrega informacao posterior ao fechamento do mes",
    "suspeita": "SUSPEITA DE VAZAMENTO: pode refletir retirada de maquina apos o fechamento",
    "identificador": "Identificador ou alta cardinalidade que nao generaliza",
    "substituida": "Substituida por feature derivada mais compacta (celulas 4 e 5)",
    "sem_sinal": "Significado desconhecido ou sem sinal, com cobertura instavel",
    "quase_constante": "Variancia quase nula ou artefato do tratamento do PIX",
    "populacao": "Constante ou redundante na populacao (todos ativos em t)",
}

EXCLUSOES = {
    **dict.fromkeys(["Status", "UF", "Rota", "qtd_stonecodes", "qtd_stonecodes_ativos",
                     "TPV_esperado_contrato", "tpv_m1", "tpv_m2", "tpv_m3",
                     "tpv_antecipado_m1", "tpv_antecipado_m2", "tpv_antecipado_m3",
                     "Receita_Liquida_Banking", "Receita_interchange",
                     "Receita_pix_in_qr_code"], "sugestao"),
    **dict.fromkeys(["dias_sem_transacionar", "Data_ultima_transacao"], "vazamento"),
    **dict.fromkeys(["Qtd_equipamentos"], "suspeita"),
    **dict.fromkeys(["Stonecode_principal", "Nome_fantasia", "Vendedor", "Cidade",
                     "canais_venda", "mcc", "rota_atual_mes"], "identificador"),
    **dict.fromkeys(["tpv_m0", "tpv_credito_m0", "tpv_debito_m0", "tpv_antecipado_m0",
                     "tpv_antecipado_auto_m0", "tpv_antecipado_spot_m0", "TPV_estimado",
                     "TPV_pix_in_qr_code", "pix_qr", "receita_pix_qr",
                     "tpv_total_m1", "tpv_total_m2", "tpv_total_m3",
                     "Data_credenciamento", "Data_primeira_ativacao",
                     "Data_primeira_transacao", "Data_assinatura_contrato",
                     "Data_fechamento_conta_stone", "Tipo_contrato",
                     "Domicilio_bancario", "Canal_venda", "Cadastro_RAV",
                     "Tipo_documento", "Mcc_key", *_DIARIO], "substituida"),
    **dict.fromkeys(["dx_m0", "duration_m0"], "sem_sinal"),
    **dict.fromkeys(["Seguro_Vida_m0", "Seguro_Loja_m0", "flag_pix_sem_equipamento",
                     "pix_qr_descartado"], "quase_constante"),
    **dict.fromkeys(["transacionou", "flag_churn", "status_movimento", "status_ba_m0",
                     "status_ba_m1", "janela_hist_completa"], "populacao"),
}

_ausentes = sorted(set(EXCLUSOES) - set(base.columns))
assert not _ausentes, f"Colunas de EXCLUSOES que nao existem na base: {_ausentes}"

print("Exclusoes por motivo:")
for chave, texto in MOTIVOS.items():
    cols = [c for c, m in EXCLUSOES.items() if m == chave]
    amostra = ", ".join(cols[:7]) + (f" ... (+{len(cols) - 7})" if len(cols) > 7 else "")
    print(f"  {len(cols):>3}  {texto}\n       {amostra}")
print(f"  {len(EXCLUSOES):>3}  total")

# -----------------------------------------------------------------------------
# Evidencia do vazamento
# -----------------------------------------------------------------------------
# A base e extraida cerca de 11 dias depois do fechamento de cada mes. A data da
# ultima transacao, e o dias_sem_transacionar calculado a partir dela, enxergam
# esses dias do mes SEGUINTE: justamente o periodo que o modelo tenta prever.
_t = base[base["conjunto"] == "treino"]
_y = _t["alvo_churn_m1"].astype(int)
_depois = _t["Data_ultima_transacao"] > _t["data_referencia"]
_ok = _t["dias_sem_transacionar"].notna()
_defasagem = (_t["dias_sem_transacionar"]
              - (_t["data_referencia"] - _t["Data_ultima_transacao"]).dt.days)

print(f"\nDefasagem mediana entre fechamento do mes e extracao: {_defasagem.median():.0f} dias")
print(f"Ultima transacao DEPOIS do fechamento do mes: {_depois.mean():.1%} das linhas")
print(f"  churn no mes seguinte entre elas: {_y[_depois].mean():.2%} "
      f"| entre as demais: {_y[~_depois].mean():.2%}")
print(f"AUC de dias_sem_transacionar sozinho: "
      f"{roc_auc_score(_y[_ok], _t.loc[_ok, 'dias_sem_transacionar']):.3f} "
      "-> bom demais para ser verdade: e o gabarito")

# -----------------------------------------------------------------------------
# Suspeita sobre Qtd_equipamentos
# -----------------------------------------------------------------------------
# Na primeira rodada do modelo, a contagem de maquinas foi a 2a feature mais
# importante. O motivo: quem VENDEU no cartao durante o mes mas aparece com ZERO
# maquinas no fechamento sai no mes seguinte na maioria das vezes. Se a retirada
# aconteceu dentro do mes, e sinal legitimo de pedido de cancelamento; se foi nos
# ~11 dias entre o fechamento e a extracao, e vazamento.
#
# O teste abaixo nao desempata de forma definitiva, mas encontra clientes so de
# maquininha, sem maquina no "fechamento", transacionando DEPOIS dele. A contagem
# nao e uma foto limpa do ultimo dia do mes, e vem da mesma extracao atrasada.
# Na duvida, sai. O custo de tirar e medido na celula 9 (sensibilidade).
_cartao = _t["tpv_m0"].fillna(0) > 0
_sem_maquina = _t["Qtd_equipamentos"].fillna(0).eq(0)
_so_maquininha = ~_t["canais_venda"].fillna("").str.contains("LINKABC|TAPONPHONE|WHATSAPPPAY")
_grupo = _cartao & _so_maquininha & _sem_maquina

print(f"\nVenderam no cartao no mes mas tem ZERO maquinas no fechamento: "
      f"{int((_cartao & _sem_maquina).sum())} linhas | churn no mes seguinte: "
      f"{_y[_cartao & _sem_maquina].mean():.1%} (com maquina: {_y[_cartao & ~_sem_maquina].mean():.1%})")
print(f"  so de maquininha (sem Link, Tap On Phone ou WhatsApp): {int(_grupo.sum())} linhas, "
      f"das quais {int((_grupo & _depois).sum())} transacionaram DEPOIS do fechamento sem ter maquina")

Exclusoes por motivo:
   15  Sugestao do usuario: dependente de outra variavel ou fora do escopo
       Status, UF, Rota, qtd_stonecodes, qtd_stonecodes_ativos, TPV_esperado_contrato, tpv_m1 ... (+8)
    2  VAZAMENTO: carrega informacao posterior ao fechamento do mes
       dias_sem_transacionar, Data_ultima_transacao
    1  SUSPEITA DE VAZAMENTO: pode refletir retirada de maquina apos o fechamento
       Qtd_equipamentos
    7  Identificador ou alta cardinalidade que nao generaliza
       Stonecode_principal, Nome_fantasia, Vendedor, Cidade, canais_venda, mcc, rota_atual_mes
   55  Substituida por feature derivada mais compacta (celulas 4 e 5)
       tpv_m0, tpv_credito_m0, tpv_debito_m0, tpv_antecipado_m0, tpv_antecipado_auto_m0, tpv_antecipado_spot_m0, TPV_estimado ... (+48)
    2  Significado desconhecido ou sem sinal, com cobertura instavel
       dx_m0, duration_m0
    4  Variancia quase nula ou artefato do tratamento do PIX
       Seguro_Vida_m0, Seguro_Loja_m0, flag_pix_sem_equ

## 4. Comportamento transacional diario

In [4]:
# =============================================================================
# 4. COMPORTAMENTO TRANSACIONAL DIARIO (tpv_d1 a tpv_d31 -> 7 features)
# =============================================================================
# O TPV diario e so cartao (debito + credito, sem PIX): a soma dos 31 dias bate com
# tpv_m0. Um cliente a caminho da saida raramente zera de uma vez; ele passa a
# vender em menos dias, abre buracos maiores e concentra a venda em poucos dias.
# As features abaixo resumem o mes nessas dimensoes:
#
#   pct_dias_com_transacao      dias com TPV > 0 / dias do mes           (frequencia)
#   dias_sem_transacao_fim_mes  dias entre o ultimo dia com TPV e o fim  (recencia)
#   maior_hiato_sem_transacao   maior sequencia de dias seguidos zerados (buracos)
#   dias_ativos_ultimos_7       dias com TPV nos 7 ultimos dias do mes   (recencia fina)
#   share_tpv_ultimos_10_dias   TPV dos 10 ultimos dias / TPV do mes     (desaceleracao)
#   cv_tpv_diario               desvio padrao / media do TPV diario      (volatilidade)
#   concentracao_tpv_top3_dias  TPV dos 3 maiores dias / TPV do mes      (concentracao)
#
# dias_sem_transacao_fim_mes e o substituto LIMPO do dias_sem_transacionar: usa so
# o que aconteceu ate o ultimo dia do mes de referencia.
#
# Convencoes:
#   - dias que nao existem no mes (tpv_d31 em abril) sao ignorados
#   - estorno (TPV diario negativo, 0,4% das linhas) conta como dia sem venda
#   - cliente ativo so por PIX nao tem venda em cartao: frequencia 0, recencia igual
#     ao tamanho do mes, e as razoes de TPV ficam nulas (nao ha denominador)
#   - linhas sem diario informado ficam com as 7 features nulas

_valores = base[_DIARIO].to_numpy(dtype=float)
_sem_diario = np.isnan(_valores).all(axis=1)
_valores = np.clip(np.nan_to_num(_valores, nan=0.0), 0, None)

_dias_mes = base["data_referencia"].dt.days_in_month.to_numpy()
_dia = np.arange(1, 32)[None, :]
_valido = _dia <= _dias_mes[:, None]
_valores = np.where(_valido, _valores, 0.0)
_venda = (_valores > 0) & _valido

# frequencia
_n_dias = _venda.sum(axis=1)

# recencia: ultimo dia com venda (0 quando nao houve venda em cartao no mes)
_ultimo_dia = np.where(_venda.any(axis=1), 31 - np.argmax(_venda[:, ::-1], axis=1), 0)

# maior sequencia de dias validos consecutivos sem venda
_seq = np.zeros(len(base))
_maior_hiato = np.zeros(len(base))
for d in range(31):
    _seq = np.where(_valido[:, d] & ~_venda[:, d], _seq + 1, 0)
    _maior_hiato = np.maximum(_maior_hiato, _seq)

# janelas do fim do mes
_ult7 = _valido & (_dia > (_dias_mes[:, None] - 7))
_ult10 = _valido & (_dia > (_dias_mes[:, None] - 10))

_total = _valores.sum(axis=1)
_com_venda = _total > 0
_den = np.where(_com_venda, _total, np.nan)

# volatilidade entre os dias validos do mes
_media = _total / _dias_mes
_variancia = np.where(_valido, (_valores - _media[:, None]) ** 2, 0.0).sum(axis=1) / _dias_mes

_diarias = pd.DataFrame({
    "pct_dias_com_transacao": _n_dias / _dias_mes,
    "dias_sem_transacao_fim_mes": _dias_mes - _ultimo_dia,
    "maior_hiato_sem_transacao": _maior_hiato,
    "dias_ativos_ultimos_7": (_venda & _ult7).sum(axis=1),
    "share_tpv_ultimos_10_dias": np.where(_ult10, _valores, 0.0).sum(axis=1) / _den,
    "cv_tpv_diario": np.sqrt(_variancia) / np.where(_com_venda, _media, np.nan),
    "concentracao_tpv_top3_dias": np.sort(_valores, axis=1)[:, -3:].sum(axis=1) / _den,
}, index=base.index).astype(float)
_diarias.loc[_sem_diario] = np.nan

base[list(_diarias.columns)] = _diarias

# Conferencia: o diario reconstruido precisa bater com o TPV de cartao do mes
_dif = (base[_DIARIO].clip(lower=0).sum(axis=1) - _total).abs().max()
print(f"Linhas sem diario informado: {int(_sem_diario.sum())} | "
      f"ativos so por PIX (sem venda em cartao): {int((~_com_venda & ~_sem_diario).sum())}")
print(f"Maior divergencia entre o diario e a reconstrucao: R$ {_dif:,.6f}")

_t = base[base["conjunto"] == "treino"]
_y = _t["alvo_churn_m1"].astype(int)
_linhas = []
for col in _diarias.columns:
    ok = _t[col].notna()
    auc = roc_auc_score(_y[ok], _t.loc[ok, col])
    _linhas.append({"feature": col,
                    "mediana_fica": _t.loc[_y.eq(0) & ok, col].median(),
                    "mediana_churna": _t.loc[_y.eq(1) & ok, col].median(),
                    "AUC": max(auc, 1 - auc),
                    "direcao": "maior -> mais churn" if auc >= 0.5 else "maior -> menos churn"})
print("\nComo cada feature separa quem churna no mes seguinte (treino):")
print(pd.DataFrame(_linhas).round(3).to_string(index=False))

print("\nCorrelacao de Spearman entre as features diarias:")
print(_t[_diarias.columns].corr(method="spearman").round(2).to_string())

# -----------------------------------------------------------------------------
# Reducao dentro do grupo diario
# -----------------------------------------------------------------------------
# As 7 medidas nao sao independentes. Nos pares acima de 0,95 as duas dizem a
# mesma coisa com formulas diferentes, e so uma fica:
#   - concentracao_tpv_top3_dias x cv_tpv_diario (0,99): a mesma volatilidade.
#     Fica o cv, que usa todos os dias do mes e nao so os tres maiores.
#   - pct_dias_com_transacao x maior_hiato_sem_transacao (-0,96): fica o hiato,
#     que separa melhor (AUC 0,89 contra 0,86) e mede IRREGULARIDADE, nao so
#     quantidade de dias. A frequencia ainda repetia dias_ativos_ultimos_7 (0,90).
REDUCAO_DIARIAS = {
    "concentracao_tpv_top3_dias": "redundante com cv_tpv_diario (Spearman 0,99)",
    "pct_dias_com_transacao": "redundante com maior_hiato_sem_transacao (-0,96) "
                              "e dias_ativos_ultimos_7 (0,90)",
}
FEATURES_DIARIAS = [c for c in _diarias.columns if c not in REDUCAO_DIARIAS]
print(f"\nFicam {len(FEATURES_DIARIAS)} das {_diarias.shape[1]} medidas: {FEATURES_DIARIAS}")

Linhas sem diario informado: 51 | ativos so por PIX (sem venda em cartao): 100
Maior divergencia entre o diario e a reconstrucao: R$ 0.000000



Como cada feature separa quem churna no mes seguinte (treino):
                   feature  mediana_fica  mediana_churna  AUC              direcao
    pct_dias_com_transacao          0.74            0.10 0.86 maior -> menos churn
dias_sem_transacao_fim_mes          0.00           13.00 0.91  maior -> mais churn
 maior_hiato_sem_transacao          2.00           18.00 0.89  maior -> mais churn
     dias_ativos_ultimos_7          5.00            0.00 0.91 maior -> menos churn
 share_tpv_ultimos_10_dias          0.32            0.00 0.77 maior -> menos churn
             cv_tpv_diario          1.24            3.70 0.83  maior -> mais churn
concentracao_tpv_top3_dias          0.38            1.00 0.83  maior -> mais churn

Correlacao de Spearman entre as features diarias:
                            pct_dias_com_transacao  dias_sem_transacao_fim_mes  maior_hiato_sem_transacao  dias_ativos_ultimos_7  share_tpv_ultimos_10_dias  cv_tpv_diario  concentracao_tpv_top3_dias
pct_dias_com_transacao

## 5. Features derivadas e categoricas padronizadas

In [5]:
# =============================================================================
# 5. FEATURES DERIVADAS E PADRONIZACAO DE CATEGORICAS
# =============================================================================
# Troca grupos de colunas correlacionadas por razoes que carregam o mesmo sinal
# em menos dimensoes, converte datas em tempo decorrido e padroniza categoricas
# que vinham com grafias misturadas (Franquia/FRANQUIA, Auto/Automatica, CPF/PF).
#
# Datas so contam se ja tinham acontecido no fechamento do mes: parte das
# assinaturas de contrato aparece com data posterior ao mes de referencia (a
# mesma extracao tardia do vazamento da celula 3) e e tratada como ausente.

def _razao(numerador, denominador):
    return numerador / denominador.where(denominador > 0)

def _meses_entre(inicio, fim):
    meses = (fim.dt.year - inicio.dt.year) * 12 + (fim.dt.month - inicio.dt.month)
    return meses.where(inicio <= fim).astype(float)      # data no futuro -> ausente

_ref = base["data_referencia"]

# ---- calendario e ciclo de vida -------------------------------------------------
# mes do ano: sazonalidade das rotas do Ceara e do litoral (alta em dez, jan e jul;
# queda depois do carnaval), que explica o churn de fev e mar
base["mes_calendario"] = _ref.dt.month.astype("int8")
base["tempo_ativacao_meses"] = _meses_entre(base["Data_primeira_ativacao"], _ref)

_multa = base["Tipo_contrato"].isin(["com_multa", "contract", "term"])
_assinado = (base["Data_assinatura_contrato"].isna()
             | (base["Data_assinatura_contrato"] <= _ref))
base["tem_contrato_multa"] = (_multa & _assinado).astype("int8")
base["conta_stone_fechada"] = (base["Data_fechamento_conta_stone"] <= _ref).astype("int8")

# ---- tendencia do TPV -------------------------------------------------------------
# razao < 1: o mes ficou abaixo da media dos 3 anteriores. Nula para quem estava
# zerado nos 3 meses (novo ativo), que nao tem base de comparacao.
base["razao_tpv_m0_media_m1_m3"] = _razao(base["tpv_total"], base["tpv_total_medio_m1_m3"])
base["meses_ativos_m1_m3"] = (base[["tpv_total_m1", "tpv_total_m2", "tpv_total_m3"]]
                              .gt(0).sum(axis=1).astype("int8"))

# ---- composicao e perfil do TPV ---------------------------------------------------
base["share_pix"] = _razao(base["pix_qr"], base["tpv_total"])
base["share_credito"] = _razao(base["tpv_credito_m0"], base["tpv_m0"])
base["share_antecipado"] = _razao(base["tpv_antecipado_m0"], base["tpv_credito_m0"])
base["ticket_medio"] = _razao(base["tpv_m0"], base["transacoes"])
# volume realizado contra o compromisso de volume declarado no credenciamento
base["aderencia_tpv_estimado"] = _razao(base["tpv_total"], base["TPV_estimado"])

# ---- categoricas padronizadas -----------------------------------------------------
def _norm(serie):
    return (serie.astype("string").str.normalize("NFKD")
            .str.encode("ascii", "ignore").str.decode("ascii")
            .str.upper().str.replace(r"[^A-Z]", "", regex=True))

base["tipo_documento"] = _norm(base["Tipo_documento"]).map(
    {"CNPJ": "CNPJ", "PJ": "CNPJ", "CPF": "CPF", "PF": "CPF", "MEI": "MEI"}
).fillna("OUTROS").astype(str)

base["canal_venda"] = _norm(base["Canal_venda"]).map(
    {"FRANQUIA": "FRANQUIA", "COMERCIAL": "COMERCIAL", "INBOUND": "INBOUND",
     "POLO": "POLO", "POLOPROPRIO": "POLO", "AUTOCREDENCIAMENTO": "AUTOCREDENCIAMENTO"}
).fillna("OUTROS").astype(str)

base["cadastro_rav"] = _norm(base["Cadastro_RAV"]).map(
    {"AUTO": "AUTO", "AUTOMATICA": "AUTO", "SPOT": "SPOT", "PONTUAL": "SPOT", "FAST": "FAST"}
).fillna("SEM_RAV").astype(str)

# recebe na propria Stone: cliente mais integrado ao ecossistema
base["domicilio_stone"] = base["Domicilio_bancario"].isin(
    ["Stone Pagamentos S.A.", "Conta Stone"]).astype("int8")

# MCC: 160 codigos entre os ativos. Os mais frequentes, contados em DOCUMENTOS no
# periodo de treino, ficam; o resto vira OUTROS para nao fragmentar o modelo.
N_MCC = 15
_docs_mcc = (base[base["conjunto"] == "treino"]
             .drop_duplicates("Documento", keep="last")["Mcc_key"].value_counts())
_top_mcc = set(_docs_mcc.head(N_MCC).index)
base["mcc_grupo"] = np.where(base["Mcc_key"].isin(_top_mcc),
                             "MCC_" + base["Mcc_key"].astype(str), "OUTROS")

# share_antecipado e calculada so para registrar por que sai: o TPV antecipado do
# mes supera o TPV de credito do mesmo mes na mediana (antecipa recebiveis de
# parcelas de meses anteriores), entao a razao nao e uma participacao. Tambem nao
# separa churn. A modalidade de antecipacao segue no modelo via cadastro_rav.
REDUCAO_DERIVADAS = {
    "share_antecipado": "razao sem significado (antecipado > credito na mediana) "
                        "e sem sinal (AUC 0,50)",
}

FEATURES_DERIVADAS = ["mes_calendario", "tempo_ativacao_meses", "tem_contrato_multa",
                      "conta_stone_fechada", "razao_tpv_m0_media_m1_m3",
                      "meses_ativos_m1_m3", "share_pix", "share_credito",
                      "ticket_medio", "aderencia_tpv_estimado",
                      "tipo_documento", "canal_venda", "cadastro_rav",
                      "domicilio_stone", "mcc_grupo"]

print("Categoricas padronizadas (niveis e linhas):")
for col in ["tipo_documento", "canal_venda", "cadastro_rav", "mcc_grupo"]:
    vc = base[col].value_counts()
    print(f"  {col}: {len(vc)} niveis | {vc.head(6).to_dict()}")
print(f"\nMCC: os {N_MCC} codigos mantidos cobrem "
      f"{base['Mcc_key'].isin(_top_mcc).mean():.1%} das linhas")
print(f"Contratos com assinatura no futuro, tratados como sem contrato: "
      f"{int((_multa & ~_assinado).sum())}")
print(f"share_antecipado (descartada): mediana {base['share_antecipado'].median():.2f}, "
      f"acima de 1 em {(base['share_antecipado'] > 1).mean():.0%} das linhas")

_num_derivadas = [c for c in FEATURES_DERIVADAS if base[c].dtype.kind in "fi"]
print("\nResumo das derivadas numericas:")
print(base[_num_derivadas].describe(percentiles=[.5]).T[["count", "mean", "50%", "max"]]
      .round(2).to_string())

Categoricas padronizadas (niveis e linhas):
  tipo_documento: 3 niveis | {'CNPJ': 21520, 'CPF': 17136, 'MEI': 2971}
  canal_venda: 6 niveis | {'FRANQUIA': 38496, 'COMERCIAL': 1196, 'INBOUND': 1057, 'POLO': 393, 'OUTROS': 272, 'AUTOCREDENCIAMENTO': 213}
  cadastro_rav: 4 niveis | {'AUTO': 30057, 'FAST': 5856, 'SPOT': 5713, 'SEM_RAV': 1}
  mcc_grupo: 16 niveis | {'OUTROS': 15211, 'MCC_5499': 6265, 'MCC_5812': 3081, 'MCC_5411': 2989, 'MCC_5963': 2000, 'MCC_8299': 1678}

MCC: os 15 codigos mantidos cobrem 63.5% das linhas
Contratos com assinatura no futuro, tratados como sem contrato: 282
share_antecipado (descartada): mediana 1.09, acima de 1 em 57% das linhas

Resumo das derivadas numericas:
                             count   mean   50%        max
mes_calendario           41,627.00   6.37  6.00      12.00
tempo_ativacao_meses     41,599.00  24.37 20.00     111.00
tem_contrato_multa       41,627.00   0.52  1.00       1.00
conta_stone_fechada      41,627.00   0.03  0.00       1.00
razao_

## 6. Dataframe do modelo

In [6]:
# =============================================================================
# 6. DATAFRAME DO MODELO
# =============================================================================
# Recorte final: chaves + alvo + features. Toda coluna da base precisa estar em
# uma das listas abaixo ou em EXCLUSOES; se o tratamento ganhar uma coluna nova,
# esta celula para e pede a classificacao, em vez de deixa-la passar calada.

CHAVES = ["Documento", "periodo", "data_referencia", "conjunto", "alvo_churn_m1"]
AUXILIARES = ["tpv_total_prox", "presente_prox"]      # so para montar o alvo

FEATURES_BASE = ["tpv_total", "tpv_total_medio_m1_m3", "transacoes", "Mensalidade_m0", "Flag_IPV", "Receita_Bruta_Banking", "Tem_seguro",
                 "flag_multi_stonecode", "flag_novo_ativo", "flag_reativacao",
                 "flag_novo_ativo_m1", "Rota_atual"]

FEATURES = FEATURES_BASE + FEATURES_DIARIAS + FEATURES_DERIVADAS
FEATURES_CATEGORICAS = ["Rota_atual", "tipo_documento", "canal_venda", "cadastro_rav",
                        "mcc_grupo"]

# Tres fontes de exclusao: as colunas da base (celula 3) e as features criadas
# aqui que a propria auditoria mostrou redundantes (celulas 4 e 5)
DESCARTES = {**EXCLUSOES, **REDUCAO_DIARIAS, **REDUCAO_DERIVADAS}

_classificadas = set(CHAVES) | set(AUXILIARES) | set(FEATURES) | set(DESCARTES)
_sem_destino = [c for c in base.columns if c not in _classificadas]
assert not _sem_destino, f"Colunas sem destino definido: {_sem_destino}"
_duplas = set(FEATURES) & set(DESCARTES)
assert not _duplas, f"Colunas marcadas como feature E como exclusao: {_duplas}"
assert len(FEATURES) == len(set(FEATURES)), "Feature repetida na lista"

df_modelo = base[CHAVES + FEATURES].reset_index(drop=True)
for col in FEATURES_CATEGORICAS:
    df_modelo[col] = df_modelo[col].astype("category")

print(f"Base tratada : {df.shape[1]} colunas | descartadas: {len(EXCLUSOES)} da base "
      f"+ {len(REDUCAO_DIARIAS) + len(REDUCAO_DERIVADAS)} criadas e reprovadas na auditoria")
print(f"Modelo       : {len(FEATURES)} features "
      f"({len(FEATURES_BASE)} originais, {len(FEATURES_DIARIAS)} do diario, "
      f"{len(FEATURES_DERIVADAS)} derivadas) + {len(CHAVES)} chaves")
print(f"Linhas       : {len(df_modelo):,} "
      f"({(df_modelo['conjunto'] == 'treino').sum():,} treino, "
      f"{(df_modelo['conjunto'] == 'escoragem').sum():,} escoragem)")
print(f"\nFeatures originais : {FEATURES_BASE}")
print(f"Features do diario : {FEATURES_DIARIAS}")
print(f"Features derivadas : {FEATURES_DERIVADAS}")

Base tratada : 107 colunas | descartadas: 92 da base + 3 criadas e reprovadas na auditoria
Modelo       : 32 features (12 originais, 5 do diario, 15 derivadas) + 5 chaves
Linhas       : 41,627 (40,068 treino, 1,559 escoragem)

Features originais : ['tpv_total', 'tpv_total_medio_m1_m3', 'transacoes', 'Mensalidade_m0', 'Flag_IPV', 'Receita_Bruta_Banking', 'Tem_seguro', 'flag_multi_stonecode', 'flag_novo_ativo', 'flag_reativacao', 'flag_novo_ativo_m1', 'Rota_atual']
Features do diario : ['dias_sem_transacao_fim_mes', 'maior_hiato_sem_transacao', 'dias_ativos_ultimos_7', 'share_tpv_ultimos_10_dias', 'cv_tpv_diario']
Features derivadas : ['mes_calendario', 'tempo_ativacao_meses', 'tem_contrato_multa', 'conta_stone_fechada', 'razao_tpv_m0_media_m1_m3', 'meses_ativos_m1_m3', 'share_pix', 'share_credito', 'ticket_medio', 'aderencia_tpv_estimado', 'tipo_documento', 'canal_venda', 'cadastro_rav', 'domicilio_stone', 'mcc_grupo']


## 7. Auditoria das features

In [7]:
# =============================================================================
# 7. AUDITORIA DAS FEATURES E GRAVACAO
# =============================================================================
# Tres checagens antes de modelar:
#   1. nulos por feature (arvore aceita nulo; modelo linear vai exigir imputacao)
#   2. poder de separacao isolado (AUC no treino). Uma variavel sozinha com AUC
#      acima de 0,95 e sinal de vazamento, como foi o dias_sem_transacionar
#   3. pares ainda muito correlacionados (|Spearman| >= 0,85): candidatos a sair
#      numa proxima rodada de reducao

_t = df_modelo[df_modelo["conjunto"] == "treino"]
_y = _t["alvo_churn_m1"].astype(int)

_numericas = [c for c in FEATURES if c not in FEATURES_CATEGORICAS]
_linhas = []
for col in _numericas:
    x = _t[col].astype(float)
    ok = x.notna()
    auc = roc_auc_score(_y[ok], x[ok]) if x[ok].nunique() > 1 else 0.5
    _linhas.append({"feature": col, "nulos_%": round(100 * (1 - ok.mean()), 1),
                    "AUC": round(max(auc, 1 - auc), 3),
                    "direcao": "maior -> mais churn" if auc >= 0.5 else "maior -> menos churn"})
auditoria = pd.DataFrame(_linhas).sort_values("AUC", ascending=False).reset_index(drop=True)
print("Features numericas, da mais para a menos separadora (treino):")
print(auditoria.to_string(index=False))

_suspeitas = auditoria.loc[auditoria["AUC"] >= 0.95, "feature"].tolist()
print(f"\nSuspeitas de vazamento (AUC >= 0,95 sozinha): {_suspeitas or 'nenhuma'}")

print("\nCategoricas: amplitude da taxa de churn entre os niveis (treino, niveis com 300+ linhas)")
for col in FEATURES_CATEGORICAS:
    taxa = (_t.groupby(col, observed=True)["alvo_churn_m1"]
              .agg(linhas="size", taxa=lambda s: s.astype(float).mean() * 100))
    taxa = taxa[taxa["linhas"] >= 300].sort_values("taxa")
    print(f"  {col:15s} de {taxa['taxa'].iloc[0]:4.1f}% ({taxa.index[0]}) "
          f"a {taxa['taxa'].iloc[-1]:4.1f}% ({taxa.index[-1]})")

_corr = _t[_numericas].astype(float).corr(method="spearman")
_pares = [(a, b, _corr.loc[a, b]) for i, a in enumerate(_numericas)
          for b in _numericas[i + 1:] if abs(_corr.loc[a, b]) >= 0.85]
print("\nPares com |Spearman| >= 0,85:")
for a, b, v in sorted(_pares, key=lambda p: -abs(p[2])):
    print(f"  {v:+.3f}  {a}  x  {b}")
if not _pares:
    print("  nenhum")

# -----------------------------------------------------------------------------
# Gravacao
# -----------------------------------------------------------------------------
# Atalho para iterar no modelo sem refazer o tratamento. Assim como o backup do
# dashboard, e DERIVADO: a fonte de verdade continua sendo este notebook.
_saida = df_modelo.assign(periodo=df_modelo["periodo"].astype("string"))
_saida.to_parquet(caminhos.BASE_MODELO_PARQUET, index=False)
print(f"\nGravado: {caminhos.BASE_MODELO_PARQUET.relative_to(caminhos.RAIZ)} "
      f"({caminhos.BASE_MODELO_PARQUET.stat().st_size / 1024**2:,.1f} MB, "
      f"{_saida.shape[0]:,} linhas x {_saida.shape[1]} colunas)")

df_modelo.head()

Features numericas, da mais para a menos separadora (treino):


                   feature  nulos_%  AUC              direcao
dias_sem_transacao_fim_mes     0.10 0.91  maior -> mais churn
     dias_ativos_ultimos_7     0.10 0.91 maior -> menos churn
 maior_hiato_sem_transacao     0.10 0.89  maior -> mais churn
             cv_tpv_diario     0.40 0.83  maior -> mais churn
                transacoes     0.00 0.78 maior -> menos churn
 share_tpv_ultimos_10_dias     0.40 0.77 maior -> menos churn
    aderencia_tpv_estimado     0.40 0.77 maior -> menos churn
                 tpv_total     0.00 0.77 maior -> menos churn
  razao_tpv_m0_media_m1_m3     4.40 0.69 maior -> menos churn
     Receita_Bruta_Banking     0.00 0.66 maior -> menos churn
                 share_pix     0.00 0.65 maior -> menos churn
     tpv_total_medio_m1_m3     0.00 0.65 maior -> menos churn
        meses_ativos_m1_m3     0.00 0.60 maior -> menos churn
             share_credito     0.40 0.58  maior -> mais churn
      tempo_ativacao_meses     0.10 0.55 maior -> menos churn
       


Pares com |Spearman| >= 0,85:
  +0.852  maior_hiato_sem_transacao  x  cv_tpv_diario

Gravado: data\processed\base_modelo.parquet (3.7 MB, 41,627 linhas x 37 colunas)


,Documento,periodo,data_referencia,conjunto,alvo_churn_m1,tpv_total,tpv_total_medio_m1_m3,transacoes,Mensalidade_m0,Flag_IPV,Receita_Bruta_Banking,Tem_seguro,flag_multi_stonecode,flag_novo_ativo,flag_reativacao,flag_novo_ativo_m1,Rota_atual,dias_sem_transacao_fim_mes,maior_hiato_sem_transacao,dias_ativos_ultimos_7,share_tpv_ultimos_10_dias,cv_tpv_diario,mes_calendario,tempo_ativacao_meses,tem_contrato_multa,conta_stone_fechada,razao_tpv_m0_media_m1_m3,meses_ativos_m1_m3,share_pix,share_credito,ticket_medio,aderencia_tpv_estimado,tipo_documento,canal_venda,cadastro_rav,domicilio_stone,mcc_grupo
0,cnpj_1006,2024-04,2024-04-30,treino,0,"81,629.34","83,308.11",616,0.00,1.00,71.24,0,1,0,0,0,Parnaíba Centro,0.00,1.00,6.00,0.32,0.53,4,21.00,1,0,0.98,3,0.18,0.71,108.93,1.36,CNPJ,FRANQUIA,SPOT,1,MCC_5499
1,cnpj_1008,2024-04,2024-04-30,treino,0,"4,415.50","6,541.07",6,79.00,1.00,0.48,0,1,0,0,0,Parnaíba Sul,7.00,7.00,0.00,0.41,2.68,4,7.00,1,0,0.68,3,0.00,0.61,735.92,0.11,CNPJ,FRANQUIA,AUTO,1,OUTROS
2,cnpj_101,2024-04,2024-04-30,treino,0,"65,056.55","32,097.15",748,138.00,0.00,112.58,0,1,0,0,0,Parnaíba Centro,0.00,1.00,6.00,0.30,0.50,4,47.00,0,0,2.03,3,0.18,0.40,71.59,2.60,CNPJ,FRANQUIA,AUTO,1,MCC_5999
3,cnpj_1011,2024-04,2024-04-30,treino,0,"52,065.07","41,333.26",133,0.00,1.00,52.65,0,1,0,0,0,Parnaíba Sul,0.00,1.00,7.00,0.38,0.79,4,55.00,1,0,1.26,3,0.15,0.93,334.67,1.74,CNPJ,FRANQUIA,AUTO,0,OUTROS
4,cnpj_1012,2024-04,2024-04-30,treino,0,"4,444.35","5,783.31",170,98.00,0.00,4.41,0,1,0,0,0,Ceará Oeste,0.00,2.00,6.00,0.24,0.89,4,15.00,1,0,0.77,3,0.02,0.56,25.70,0.18,MEI,FRANQUIA,AUTO,1,MCC_5812


## 8. Desenho da validacao

Treino em 2025, teste em jan-jun/2026, medido pela capacidade do time: **120 clientes por mes** (200 por telefone).

In [8]:
# =============================================================================
# 8. DESENHO DA VALIDACAO
# =============================================================================
# Treino: meses de 2025 (jan a nov). Teste: jan a jun de 2026, os meses com alvo.
# Dezembro de 2025 fica de fora como intervalo de seguranca entre os dois: o alvo
# dele e o que aconteceu em jan/2026, mes em que o teste ja comeca.
#
# Divisao TEMPORAL, nunca aleatoria: o mesmo cliente aparece em varios meses, e
# embaralhar colocaria o futuro dele dentro do treino.
#
# Metrica principal: CAPACIDADE DE ABORDAGEM. O time aborda com qualidade 120
# clientes por mes (ate 200 por telefone). Todo mes, cada abordagem ranqueia os
# ativos e entrega os 120 de maior risco; mede-se quantos churns do mes seguinte
# estavam nessa lista.
#   recall@120   = churns na lista / churns do mes
#   precisao@120 = churns na lista / 120
#   lift@120     = precisao@120 / taxa de churn do mes (quantas vezes melhor que sortear)
#
# Nenhum parametro e escolhido olhando o teste: os hiperparametros sao fixos e
# conservadores, definidos antes de rodar.

from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

CAPACIDADE = 120
CAPACIDADE_TELEFONE = 200

TREINO_INICIO, TREINO_FIM = pd.Period("2025-01", "M"), pd.Period("2025-11", "M")
TESTE_INICIO, TESTE_FIM = pd.Period("2026-01", "M"), pd.Period("2026-06", "M")

_rotulado = df_modelo[df_modelo["conjunto"] == "treino"]
treino = _rotulado[(_rotulado["periodo"] >= TREINO_INICIO)
                   & (_rotulado["periodo"] <= TREINO_FIM)].reset_index(drop=True)
teste = _rotulado[(_rotulado["periodo"] >= TESTE_INICIO)
                  & (_rotulado["periodo"] <= TESTE_FIM)].reset_index(drop=True)
y_treino = treino["alvo_churn_m1"].astype(int).to_numpy()
y_teste = teste["alvo_churn_m1"].astype(int).to_numpy()

FEATURES_NUMERICAS = [c for c in FEATURES if c not in FEATURES_CATEGORICAS]

# Valores monetarios e razoes tem cauda extrema (a razao de TPV chega a 282.900).
# O modelo linear e o k-means medem distancia, entao essas variaveis entram em
# log. O gradient boosting usa os valores crus: arvore so olha a ordem.
FEATURES_ASSIMETRICAS = ["tpv_total", "tpv_total_medio_m1_m3", "transacoes",
                         "Mensalidade_m0", "Receita_Bruta_Banking",
                         "razao_tpv_m0_media_m1_m3", "ticket_medio",
                         "aderencia_tpv_estimado", "cv_tpv_diario", "tempo_ativacao_meses"]


def _log_assimetricas(x):
    x = x.astype(float).copy()
    for col in FEATURES_ASSIMETRICAS:
        x[col] = np.sign(x[col]) * np.log1p(x[col].abs())
    return x


def padronizador():
    """log nas assimetricas -> mediana nos nulos -> media 0 e desvio 1 (ajustado no treino)."""
    return make_pipeline(FunctionTransformer(_log_assimetricas),
                         SimpleImputer(strategy="median"), StandardScaler())


def avalia_capacidade(dados, score, capacidade=CAPACIDADE):
    """Ranqueia os ativos de cada mes e mede a lista dos `capacidade` de maior risco."""
    tabela = dados[["periodo", "Documento"]].assign(
        alvo=dados["alvo_churn_m1"].astype(int).to_numpy(), score=score)
    tabela["posicao"] = tabela.groupby("periodo")["score"].rank(method="first",
                                                                ascending=False)
    tabela["na_lista"] = tabela["posicao"] <= capacidade
    tabela["pego"] = tabela["alvo"] * tabela["na_lista"]
    por_mes = tabela.groupby("periodo").agg(ativos=("alvo", "size"),
                                            churns=("alvo", "sum"),
                                            pegos=("pego", "sum"))
    por_mes["recall"] = por_mes["pegos"] / por_mes["churns"]
    por_mes["precisao"] = por_mes["pegos"] / capacidade
    por_mes["lift"] = por_mes["precisao"] / (por_mes["churns"] / por_mes["ativos"])
    return por_mes, tabela


def recall_cliente(tabela):
    """Clientes que churnaram e passaram pela lista em algum mes ate o do churn."""
    churn = (tabela[tabela["alvo"] == 1].groupby("Documento")["periodo"].min()
             .rename("primeiro_churn"))
    na_lista = tabela[tabela["na_lista"]].merge(churn, left_on="Documento",
                                                right_index=True)
    avisados = na_lista.loc[na_lista["periodo"] <= na_lista["primeiro_churn"],
                            "Documento"].nunique()
    return avisados / len(churn)


def resume(nome, dados, score):
    y = dados["alvo_churn_m1"].astype(int)
    linha = {"abordagem": nome, "ROC-AUC": roc_auc_score(y, score),
             "PR-AUC": average_precision_score(y, score)}
    for cap in (CAPACIDADE, CAPACIDADE_TELEFONE):
        por_mes, tabela = avalia_capacidade(dados, score, cap)
        linha[f"recall@{cap}"] = por_mes["pegos"].sum() / por_mes["churns"].sum()
        linha[f"precisao@{cap}"] = por_mes["pegos"].sum() / (cap * len(por_mes))
        if cap == CAPACIDADE:
            linha["lift@120"] = por_mes["lift"].mean()
            linha["clientes avisados@120"] = recall_cliente(tabela)
    return linha


_mes_t = teste.groupby("periodo")["alvo_churn_m1"].agg(ativos="size", churns="sum")
print(f"Treino: {len(treino):,} linhas ({TREINO_INICIO} a {TREINO_FIM}), "
      f"{y_treino.sum():,} churns ({y_treino.mean():.2%})")
print(f"Teste : {len(teste):,} linhas ({TESTE_INICIO} a {TESTE_FIM}), "
      f"{y_teste.sum():,} churns ({y_teste.mean():.2%})")
print(f"\nNo teste, o mes medio tem {_mes_t['ativos'].mean():,.0f} ativos e "
      f"{_mes_t['churns'].mean():,.0f} churns no mes seguinte.")
print(f"A lista de {CAPACIDADE} cobre {CAPACIDADE / _mes_t['ativos'].mean():.1%} da base; "
      f"a de {CAPACIDADE_TELEFONE}, {CAPACIDADE_TELEFONE / _mes_t['ativos'].mean():.1%}.")

Treino: 16,520 linhas (2025-01 a 2025-11), 950 churns (5.75%)
Teste : 8,962 linhas (2026-01 a 2026-06), 561 churns (6.26%)

No teste, o mes medio tem 1,494 ativos e 94 churns no mes seguinte.
A lista de 120 cobre 8.0% da base; a de 200, 13.4%.


## 9. Regra simples, logistica e gradient boosting

In [9]:
# =============================================================================
# 9. TRES ABORDAGENS COMPARADAS NA CAPACIDADE DO TIME
# =============================================================================
#   Regra simples     ranqueia por dias sem venda no fim do mes. E a referencia:
#                     um modelo so se justifica se pegar mais churns que ela.
#   Logistica         linear, so variaveis numericas, classes balanceadas.
#   Gradient boosting arvores, todas as 33 features, nulos e categoricas nativos.
#
# Sorteio: o que uma lista aleatoria de 120 pegaria, calculado analiticamente.

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

SCORES = {}

# ---- regra simples ------------------------------------------------------------
def score_regra(dados):
    """Dias sem venda no fim do mes; desempate por menos dias ativos na ultima
    semana e depois pelo maior hiato."""
    return (dados["dias_sem_transacao_fim_mes"].fillna(31)
            + 0.01 * (7 - dados["dias_ativos_ultimos_7"].fillna(0))
            + 0.0001 * dados["maior_hiato_sem_transacao"].fillna(31)).to_numpy()


SCORES["Regra: dias sem venda no fim do mes"] = score_regra(teste)

# ---- regressao logistica ------------------------------------------------------
modelo_logistica = make_pipeline(padronizador(),
                                 LogisticRegression(max_iter=2000, class_weight="balanced"))
modelo_logistica.fit(treino[FEATURES_NUMERICAS], y_treino)
SCORES["Regressao logistica"] = modelo_logistica.predict_proba(teste[FEATURES_NUMERICAS])[:, 1]

# ---- gradient boosting --------------------------------------------------------
# Sem early stopping: ele separaria a validacao interna ao acaso, misturando
# meses do mesmo cliente. Sem class_weight: a probabilidade sai calibrada e pode
# ser lida como risco.
PARAMS_BOOSTING = dict(learning_rate=0.05, max_iter=300, max_leaf_nodes=31,
                       min_samples_leaf=40, l2_regularization=1.0,
                       categorical_features="from_dtype", early_stopping=False,
                       random_state=0)


def boosting():
    return HistGradientBoostingClassifier(**PARAMS_BOOSTING)


modelo_boosting = boosting().fit(treino[FEATURES], y_treino)
SCORES["Gradient boosting"] = modelo_boosting.predict_proba(teste[FEATURES])[:, 1]

# ---- comparacao ---------------------------------------------------------------
comparacao = pd.DataFrame([resume(nome, teste, s) for nome, s in SCORES.items()])

_sorteio = {"abordagem": "Sorteio (referencia)", "ROC-AUC": 0.5, "PR-AUC": y_teste.mean()}
for cap in (CAPACIDADE, CAPACIDADE_TELEFONE):
    _fracao = np.minimum(1, cap / _mes_t["ativos"])
    _sorteio[f"recall@{cap}"] = (_mes_t["churns"] * _fracao).sum() / _mes_t["churns"].sum()
    _sorteio[f"precisao@{cap}"] = (_mes_t["churns"] * _fracao).sum() / (cap * len(_mes_t))
_sorteio["lift@120"] = 1.0
comparacao = pd.concat([comparacao, pd.DataFrame([_sorteio])]).set_index("abordagem")

_formato = comparacao.copy()
for col in _formato.columns:
    if col.startswith(("recall", "precisao", "clientes")):
        _formato[col] = (_formato[col] * 100).map(lambda v: "-" if pd.isna(v) else f"{v:.1f}%")
    else:
        _formato[col] = _formato[col].map(lambda v: "-" if pd.isna(v) else f"{v:.3f}")
print("Teste jan-jun/2026, somando os 6 meses:")
print(_formato.to_string())

_melhor = comparacao.drop(index="Sorteio (referencia)")["recall@120"].idxmax()
por_mes_melhor, tabela_melhor = avalia_capacidade(teste, SCORES[_melhor])
print(f"\nMes a mes, lista de {CAPACIDADE} do melhor ({_melhor}):")
print(por_mes_melhor.assign(recall=lambda x: (x["recall"] * 100).round(1),
                           precisao=lambda x: (x["precisao"] * 100).round(1),
                           lift=lambda x: x["lift"].round(1)).to_string())

# ---- sensibilidade: o custo de ter tirado Qtd_equipamentos (celula 3) ---------
# Mesmo boosting, com a variavel suspeita de volta. So para medir; nao entra na lista.
_equip = base[["Documento", "periodo", "Qtd_equipamentos"]]
_treino_eq = treino.merge(_equip, on=["Documento", "periodo"], how="left")
_teste_eq = teste.merge(_equip, on=["Documento", "periodo"], how="left")
_score_eq = (boosting().fit(_treino_eq[FEATURES + ["Qtd_equipamentos"]], y_treino)
             .predict_proba(_teste_eq[FEATURES + ["Qtd_equipamentos"]])[:, 1])
sensibilidade_equip = resume("Boosting COM Qtd_equipamentos", teste, _score_eq)
print(f"\nSensibilidade: com Qtd_equipamentos o boosting teria recall@120 de "
      f"{sensibilidade_equip['recall@120']:.1%} e PR-AUC de {sensibilidade_equip['PR-AUC']:.3f} "
      f"(sem ela: {comparacao.loc['Gradient boosting', 'recall@120']:.1%} e "
      f"{comparacao.loc['Gradient boosting', 'PR-AUC']:.3f})")

_regra = comparacao.loc["Regra: dias sem venda no fim do mes"]
_boost = comparacao.loc["Gradient boosting"]
_churns_mes = _mes_t["churns"].mean()
print(f"\nGanho do boosting sobre a regra na lista de {CAPACIDADE}: "
      f"{(_boost['recall@120'] - _regra['recall@120']) * 100:+.1f} pontos de recall, "
      f"cerca de {(_boost['recall@120'] - _regra['recall@120']) * _churns_mes:+.0f} "
      f"churns a mais por mes na lista")

Teste jan-jun/2026, somando os 6 meses:
                                    ROC-AUC PR-AUC recall@120 precisao@120 lift@120 clientes avisados@120 recall@200 precisao@200
abordagem                                                                                                                        
Regra: dias sem venda no fim do mes   0.930  0.435      59.5%        46.4%    7.534                 64.2%      77.4%        36.2%
Regressao logistica                   0.913  0.532      62.4%        48.6%    7.917                 68.3%      79.5%        37.2%
Gradient boosting                     0.938  0.543      62.6%        48.8%    7.933                 67.5%      79.1%        37.0%
Sorteio (referencia)                  0.500  0.063       8.0%         6.3%    1.000                     -      13.4%         6.3%

Mes a mes, lista de 120 do melhor (Gradient boosting):
         ativos  churns  pegos  recall  precisao  lift
periodo                                               
2026-01    152


Sensibilidade: com Qtd_equipamentos o boosting teria recall@120 de 64.7% e PR-AUC de 0.621 (sem ela: 62.6% e 0.543)

Ganho do boosting sobre a regra na lista de 120: +3.0 pontos de recall, cerca de +3 churns a mais por mes na lista


## 10. Importancia das features

In [10]:
# =============================================================================
# 10. O QUE O GRADIENT BOOSTING USA
# =============================================================================
# Importancia por permutacao, medida no TESTE: embaralha uma feature por vez e
# mede quanto o PR-AUC cai. Queda grande = o modelo depende dela para acertar.
#
# Leitura com cuidado: features correlacionadas dividem a importancia entre si
# (maior_hiato e cv_tpv_diario tem Spearman 0,85). Uma feature com queda perto de
# zero pode estar coberta por outra, e nao necessariamente ser inutil.

from sklearn.inspection import permutation_importance

_perm = permutation_importance(modelo_boosting, teste[FEATURES], y_teste,
                               scoring="average_precision", n_repeats=5, random_state=0)
importancia = (pd.DataFrame({"feature": FEATURES,
                             "queda_PR_AUC": _perm.importances_mean,
                             "desvio": _perm.importances_std})
               .sort_values("queda_PR_AUC", ascending=False)
               .reset_index(drop=True))

_base_pr = comparacao.loc["Gradient boosting", "PR-AUC"]
importancia["queda_%"] = importancia["queda_PR_AUC"] / _base_pr * 100
print(f"PR-AUC do boosting no teste: {_base_pr:.3f}\n")
print(importancia.round({"queda_PR_AUC": 4, "desvio": 4, "queda_%": 1}).to_string(index=False))

_irrelevantes = importancia.loc[importancia["queda_PR_AUC"] <= importancia["desvio"], "feature"]
print(f"\nFeatures cuja queda nao supera o proprio ruido ({len(_irrelevantes)}): "
      f"{', '.join(_irrelevantes)}")

PR-AUC do boosting no teste: 0.543

                   feature  queda_PR_AUC  desvio  queda_%
dias_sem_transacao_fim_mes          0.31    0.01    58.00
     dias_ativos_ultimos_7          0.07    0.00    12.90
 maior_hiato_sem_transacao          0.06    0.01    12.00
                 share_pix          0.04    0.00     8.20
             cv_tpv_diario          0.03    0.01     6.00
                transacoes          0.02    0.00     3.50
     tpv_total_medio_m1_m3          0.01    0.01     2.60
  razao_tpv_m0_media_m1_m3          0.01    0.00     2.50
        meses_ativos_m1_m3          0.01    0.00     2.10
     Receita_Bruta_Banking          0.01    0.00     2.10
                 mcc_grupo          0.01    0.01     1.90
    aderencia_tpv_estimado          0.01    0.01     1.50
                Rota_atual          0.01    0.00     1.40
            Mensalidade_m0          0.00    0.00     0.90
              ticket_medio          0.00    0.00     0.80
                  Flag_IPV          

## 11. Clusters de risco

In [11]:
# =============================================================================
# 11. CLUSTERS DE RISCO: QUEM SAO OS CLIENTES DA LISTA
# =============================================================================
# O boosting diz QUEM priorizar; os clusters dizem POR QUE e que tipo de acao cabe.
#
# O k-means agrupa TODOS os ativos de 2025, e nao so quem saiu: agrupar so os
# churners descreve como eles sao, mas nao o que os separa de quem fica (no teste
# exploratorio, essa versao teve AUC 0,54, praticamente um sorteio).
#
# Os clusters sao numerados por risco: C1 e o de maior taxa de churn no treino.
# Sao ajustados so em 2025 e aplicados a 2026 sem reajuste, para medir se o
# risco de cada grupo se mantem de um ano para o outro.

from sklearn.cluster import KMeans

N_CLUSTERS = 6

padronizador_cluster = padronizador().fit(treino[FEATURES_NUMERICAS])
modelo_kmeans = KMeans(N_CLUSTERS, n_init=10, random_state=0).fit(
    padronizador_cluster.transform(treino[FEATURES_NUMERICAS]))

_taxa_treino = (pd.Series(y_treino).groupby(modelo_kmeans.labels_).mean()
                .sort_values(ascending=False))
ROTULO_CLUSTER = {int(c): f"C{i + 1}" for i, c in enumerate(_taxa_treino.index)}


def atribui_cluster(dados):
    rotulos = modelo_kmeans.predict(padronizador_cluster.transform(dados[FEATURES_NUMERICAS]))
    return pd.Series(rotulos, index=dados.index).map(ROTULO_CLUSTER)


treino["cluster"] = atribui_cluster(treino)
teste["cluster"] = atribui_cluster(teste)

# ---- perfil -------------------------------------------------------------------
_medianas = {"tpv_total": "TPV mediano", "dias_sem_transacao_fim_mes": "dias s/ venda fim",
             "maior_hiato_sem_transacao": "maior hiato", "dias_ativos_ultimos_7": "dias ativos ult7",
             "meses_ativos_m1_m3": "meses ativos 3ant", "tempo_ativacao_meses": "meses desde ativ",
             "share_pix": "share PIX"}
perfil_clusters = (
    treino.groupby("cluster")
    .agg(linhas_2025=("alvo_churn_m1", "size"),
         churn_2025=("alvo_churn_m1", lambda s: s.astype(float).mean() * 100),
         novo_ativo=("flag_novo_ativo", lambda s: s.mean() * 100),
         reativacao=("flag_reativacao", lambda s: s.mean() * 100))
    .join(treino.groupby("cluster")[list(_medianas)].median().rename(columns=_medianas))
    .join(teste.groupby("cluster")
          .agg(churn_2026=("alvo_churn_m1", lambda s: s.astype(float).mean() * 100),
               parte_churns_2026=("alvo_churn_m1", lambda s: s.sum() / y_teste.sum() * 100),
               parte_base_2026=("alvo_churn_m1", lambda s: len(s) / len(teste) * 100)))
)
print("Perfil dos clusters (medianas no treino; % de novo ativo e reativacao no treino):")
print(perfil_clusters.round(1).to_string())

# ---- a lista de 120 do boosting, vista pelos clusters -------------------------
_tab = tabela_melhor.assign(cluster=teste["cluster"].to_numpy())
composicao = (
    _tab.groupby("cluster")
    .agg(churns=("alvo", "sum"),
         na_lista_por_mes=("na_lista", lambda s: s.sum() / _tab["periodo"].nunique()),
         churns_pegos=("pego", "sum"))
)
composicao["churns_perdidos"] = composicao["churns"] - composicao["churns_pegos"]
composicao["recall_%"] = composicao["churns_pegos"] / composicao["churns"] * 100
print(f"\nLista de {CAPACIDADE} do melhor modelo no teste, por cluster:")
print(composicao.round(1).to_string())
print(f"\nChurns que a lista NAO pegou: {int(composicao['churns_perdidos'].sum())}, "
      "distribuidos acima por cluster.")

Perfil dos clusters (medianas no treino; % de novo ativo e reativacao no treino):
         linhas_2025  churn_2025  novo_ativo  reativacao  TPV mediano  dias s/ venda fim  maior hiato  dias ativos ult7  meses ativos 3ant  meses desde ativ  share PIX  churn_2026  parte_churns_2026  parte_base_2026
cluster                                                                                                                                                                                                                
C1               277       25.30        0.00      100.00     1,297.00               5.00        18.00              1.00               2.00             19.00       0.00       34.50               9.10             1.70
C2              4576       13.60        0.00        0.00     2,165.00               3.00        10.00              1.00               3.00             22.00       0.00       15.90              64.50            25.40
C3               672       12.10      100.00        0.

### Leitura dos clusters (execucao de 12/09/2026)

| Cluster | Quem e | Churn 2025 -> 2026 | Base / churns de 2026 |
|---|---|---|---|
| **C1** | Reativados: voltaram a vender depois de um mes zerado. Pequenos (TPV R$ 1,3 mil), hiato de 18 dias | 25,3% -> **34,5%** | 1,7% / 9,1% |
| **C2** | Pequenos e irregulares: TPV R$ 2,2 mil, 3 dias sem venda no fim do mes, hiato de 10 dias | 13,6% -> 15,9% | 25,4% / **64,5%** |
| **C3** | Novos ativos, no primeiro mes de venda | 12,1% -> 11,0% | 4,7% / 8,2% |
| **C4** | Recem-chegados de TPV alto (R$ 13 mil), so 1 mes ativo nos 3 anteriores | 7,3% -> **10,7%** | 4,2% / 7,1% |
| **C5** | Carteira madura media: TPV R$ 18 mil, vende todo dia | 1,3% -> 1,1% | 50,2% / 8,7% |
| **C6** | Carteira madura grande e antiga: TPV R$ 30 mil, 52 meses de casa | 1,1% -> 1,0% | 13,9% / 2,3% |

- O risco de cada grupo se manteve de um ano para o outro, exceto **reativados (C1)** e
  **recem-chegados de TPV alto (C4)**, que pioraram em 2026.
- A lista de 120 e dominada pelo C2: 81 por mes no teste.
- Os churns que escapam da lista vem sobretudo do C2 (128 no semestre) e do C5 (27): no C5 sao
  clientes que vendiam normalmente no mes anterior, sem sinal para o modelo captar.

## 12. Lista de risco do proximo mes

In [12]:
# =============================================================================
# 12. LISTA DE RISCO DO PROXIMO MES
# =============================================================================
# O modelo final e o mesmo gradient boosting, com os mesmos parametros, retreinado
# com TODOS os meses rotulados (abr/2024 a jun/2026). Escora os clientes ativos no
# ultimo mes da base e entrega a lista de abordagem do mes seguinte:
#   posicoes 1 a 120    abordagem qualificada
#   posicoes 121 a 200  telefone
#
# Os documentos sao anonimizados. A lista e gravada em data/processed/, fora do Git.

_rotulado = df_modelo[df_modelo["conjunto"] == "treino"]
modelo_final = boosting().fit(_rotulado[FEATURES], _rotulado["alvo_churn_m1"].astype(int))

MES_ALVO = ULTIMO_MES + 1
escoragem = df_modelo[df_modelo["conjunto"] == "escoragem"].copy()
escoragem["risco"] = modelo_final.predict_proba(escoragem[FEATURES])[:, 1]
escoragem["cluster"] = atribui_cluster(escoragem)
escoragem["posicao"] = escoragem["risco"].rank(method="first", ascending=False).astype(int)

lista_risco = (escoragem[escoragem["posicao"] <= CAPACIDADE_TELEFONE]
               .sort_values("posicao")
               .assign(faixa=lambda x: np.where(x["posicao"] <= CAPACIDADE,
                                                "abordagem qualificada", "telefone"))
               [["posicao", "faixa", "Documento", "Rota_atual", "cluster", "risco",
                 "tpv_total", "tpv_total_medio_m1_m3", "dias_sem_transacao_fim_mes",
                 "maior_hiato_sem_transacao", "dias_ativos_ultimos_7",
                 "tempo_ativacao_meses", "flag_novo_ativo", "flag_reativacao"]]
               .reset_index(drop=True))

_top = lista_risco[lista_risco["faixa"] == "abordagem qualificada"]
_precisao_teste = comparacao.loc["Gradient boosting", "precisao@120"]
print(f"Escorados: {len(escoragem):,} clientes ativos em {ULTIMO_MES}; lista para {MES_ALVO}")
print(f"Churns esperados entre os {CAPACIDADE} (soma das probabilidades): "
      f"{_top['risco'].sum():,.0f}")
print(f"Referencia do teste: a lista de {CAPACIDADE} teve precisao de {_precisao_teste:.0%}, "
      f"ou cerca de {_precisao_teste * CAPACIDADE:,.0f} churns reais por mes")
print(f"Risco na lista: de {_top['risco'].min():.0%} (posicao {CAPACIDADE}) "
      f"a {_top['risco'].max():.0%} (posicao 1); mediana da base: "
      f"{escoragem['risco'].median():.1%}")

print(f"\nOs {CAPACIDADE} de abordagem qualificada, por rota e cluster:")
print(pd.crosstab(_top["Rota_atual"], _top["cluster"], margins=True, margins_name="Total")
        .to_string())

CAMINHO_LISTA = caminhos.DADOS_TRATADOS / f"lista_risco_{MES_ALVO}.parquet"
lista_risco.to_parquet(CAMINHO_LISTA, index=False)
print(f"\nGravado: {CAMINHO_LISTA.relative_to(caminhos.RAIZ)} ({len(lista_risco)} clientes)")

lista_risco.head(15).assign(risco=lambda x: (x["risco"] * 100).round(1))

Escorados: 1,559 clientes ativos em 2026-07; lista para 2026-08
Churns esperados entre os 120 (soma das probabilidades): 56
Referencia do teste: a lista de 120 teve precisao de 49%, ou cerca de 58 churns reais por mes
Risco na lista: de 19% (posicao 120) a 96% (posicao 1); mediana da base: 0.1%

Os 120 de abordagem qualificada, por rota e cluster:
cluster          C1  C2  C3  C4  C5  C6  Total
Rota_atual                                    
Ceará Leste       0  15   4   1   1   0     21
Ceará Oeste       1  17   2   0   0   0     20
Interior PI       0   4   0   1   0   0      5
Nao informada     0   0   2   0   0   0      2
Outras Rotas      0   1   0   0   0   0      1
Parnaíba Centro   2  16   3   2   1   0     24
Parnaíba Sul      0  26   4   1   3   0     34
Área Litoral PI   1   9   2   0   0   1     13
Total             4  88  17   5   5   1    120

Gravado: data\processed\lista_risco_2026-08.parquet (200 clientes)


,posicao,faixa,Documento,Rota_atual,cluster,risco,tpv_total,tpv_total_medio_m1_m3,dias_sem_transacao_fim_mes,maior_hiato_sem_transacao,dias_ativos_ultimos_7,tempo_ativacao_meses,flag_novo_ativo,flag_reativacao
0,1,abordagem qualificada,cnpj_2019,Ceará Leste,C2,96.20,"7,074.10","30,679.27",16.00,16.00,0.00,16.00,0,0
1,2,abordagem qualificada,cnpj_1523,Parnaíba Sul,C2,95.30,290.48,"62,553.90",29.00,29.00,0.00,4.00,0,0
2,3,abordagem qualificada,cnpj_1552,Ceará Leste,C5,90.80,"15,304.11","44,636.30",3.00,3.00,4.00,10.00,0,0
3,4,abordagem qualificada,cpf_527,Parnaíba Centro,C4,88.60,210.80,"4,460.27",26.00,26.00,0.00,1.00,0,0
4,5,abordagem qualificada,cnpj_3128,Ceará Leste,C2,88.60,"1,769.91","44,173.82",24.00,24.00,0.00,8.00,0,0
5,6,abordagem qualificada,cnpj_2212,Parnaíba Centro,C2,88.30,29.90,"3,035.59",29.00,29.00,0.00,24.00,0,0
6,7,abordagem qualificada,cpf_2866,Ceará Leste,C2,87.50,214.72,"37,401.55",21.00,21.00,0.00,4.00,0,0
7,8,abordagem qualificada,cnpj_2848,Parnaíba Centro,C2,84.90,"12,843.74","36,150.27",25.00,25.00,0.00,3.00,0,0
8,9,abordagem qualificada,cnpj_1320,Parnaíba Centro,C4,84.30,56.00,11.67,29.00,29.00,0.00,20.00,0,0
9,10,abordagem qualificada,cnpj_2051,Ceará Leste,C2,83.50,30.50,"40,073.16",26.00,26.00,0.00,17.00,0,0


## 13. Validacao com o mes realizado

Confere a lista da celula 12 contra a carteira do mes seguinte, quando ela chega em `data/raw/`.

In [13]:
# =============================================================================
# 13. VALIDACAO COM O MES REALIZADO
# =============================================================================
# Quando chega a carteira do mes para o qual a lista foi feita, esta celula mede
# se a lista acertou. Usa a carteira JA PSEUDONIMIZADA, gerada por
# src/pseudonimizar_mes.py:
#   data/raw/base_<AAAA-MM>.parquet
# Os pseudonimos sao os mesmos da base do projeto, entao o cruzamento e direto e a
# celula nao precisa de nenhum de-para nem de dado real. Sem o arquivo do mes, a
# celula so avisa e segue.
#
# Churn no mes realizado segue a regra do projeto: TPV Total (cartao + PIX QR so
# com maquina instalada) zerado para quem estava ativo no mes escorado. Cliente
# ausente do arquivo fica como indefinido, como no treino.
#
# A carteira pseudonimizada tem as 90 colunas da base. Duas conferencias feitas em
# ago/2026 com a carteira bruta (a marcacao churn_meta da Stone e a data de
# extracao) so rodam se essas colunas existirem no arquivo; o resultado delas esta
# registrado no README (secao 11.9).
#
# ATENCAO: a validacao so e limpa enquanto ninguem age sobre a lista. Quando o
# time abordar os clientes, quem for retido aparece como "alarme falso" e o
# modelo parece pior justamente quando funciona. A partir dai, sorteie um grupo
# de controle dentro da lista, que nao e abordado, e meca o acerto nele.


def arquivo_realizado(periodo):
    caminho = caminhos.DADOS_BRUTOS / f"base_{periodo}.parquet"
    return caminho if caminho.exists() else None


def carrega_realizado(arquivo):
    """TPV Total do mes realizado por cliente pseudonimizado, com a regra de PIX do projeto."""
    bruto = pd.read_parquet(arquivo)
    com_maquina = bruto["Qtd_equipamentos"].fillna(0) > 0
    bruto["tpv_total_realizado"] = (bruto["tpv_m0"].fillna(0)
                                    + bruto["TPV_pix_in_qr_code"].fillna(0).where(com_maquina, 0.0))
    agregacoes = {"tpv_total_realizado": ("tpv_total_realizado", "sum"),
                  "tpv_cartao_mes_anterior": ("tpv_m1", "sum")}
    if "churn_meta" in bruto.columns:
        agregacoes["churn_meta"] = ("churn_meta", "max")
    por_documento = bruto.groupby("Documento").agg(**agregacoes)
    info = {"linhas": len(bruto),
            "referencia": pd.to_datetime(bruto["data_referencia"]).iloc[0],
            "atualizacao": (pd.to_datetime(bruto["data_ultima_atualizacao"]).iloc[0]
                            if "data_ultima_atualizacao" in bruto.columns else None)}
    return por_documento, info


def _pct(v):
    return f"{v * 100:.1f}%"


def valida_lista(realizado, info):
    """Compara a lista de risco (celula 12) com o churn que de fato aconteceu."""
    assert info["referencia"].to_period("M") == MES_ALVO, (
        f"o arquivo e de {info['referencia']:%m/%Y}, mas a lista e para {MES_ALVO}")
    atualizacao = (f" | atualizada em {info['atualizacao']:%d/%m/%Y}, "
                   f"{(info['atualizacao'] - info['referencia']).days} dias apos o fechamento"
                   if info["atualizacao"] is not None else "")
    print(f"Carteira de {MES_ALVO}: {info['linhas']:,} linhas{atualizacao}")

    # conferencia do cruzamento: o tpv_m1 do mes realizado e o TPV de cartao do mes escorado
    _mes_escorado = (df.loc[df["periodo"] == ULTIMO_MES, ["Documento", "tpv_m0"]]
                       .set_index("Documento")["tpv_m0"])
    _conf = realizado.join(_mes_escorado, how="inner")
    _bate = ((_conf["tpv_cartao_mes_anterior"] - _conf["tpv_m0"]).abs() <= 1).mean()
    print(f"Conferencia do cruzamento: TPV de {ULTIMO_MES} bate em {_pct(_bate)} "
          f"dos {len(_conf):,} clientes presentes nos dois meses")
    assert _bate > 0.95, "cruzamento suspeito: confira a pseudonimizacao do mes"

    aval = escoragem.merge(realizado, left_on="Documento", right_index=True, how="left")
    aval["presente"] = aval["tpv_total_realizado"].notna()
    aval["churn_realizado"] = (aval["presente"] & (aval["tpv_total_realizado"] <= 0)).astype(int)
    aval["na_lista"] = aval["posicao"] <= CAPACIDADE
    aval["score_regra"] = score_regra(aval)

    d = aval[aval["presente"]]
    y = d["churn_realizado"].to_numpy()
    print(f"\nAtivos em {ULTIMO_MES}: {len(aval):,} | ausentes do arquivo (indefinidos): "
          f"{int((~aval['presente']).sum())} | churn em {MES_ALVO}: {y.sum()} ({_pct(y.mean())})")
    if "churn_meta" in d.columns:
        print("\nDefinicao do projeto x churn_meta da Stone:")
        print(pd.crosstab(d["churn_realizado"], d["churn_meta"], rownames=["churn_projeto"],
                          colnames=["churn_meta"], margins=True).to_string())

    # ---- acerto da lista ----------------------------------------------------------
    linhas = []
    for nome, score in [("Gradient boosting (lista entregue)", d["risco"].to_numpy()),
                        ("Regra: dias sem venda no fim do mes", d["score_regra"].to_numpy())]:
        rank = pd.Series(score).rank(method="first", ascending=False).to_numpy()
        linha = {"abordagem": nome, "ROC-AUC": roc_auc_score(y, score),
                 "PR-AUC": average_precision_score(y, score)}
        for cap in (CAPACIDADE, CAPACIDADE_TELEFONE):
            pegos = int(y[rank <= cap].sum())
            linha[f"churns@{cap}"] = pegos
            linha[f"recall@{cap}"] = pegos / y.sum()
            linha[f"precisao@{cap}"] = pegos / int((rank <= cap).sum())
        linhas.append(linha)
    linhas.append({"abordagem": "Sorteio (esperado)", "ROC-AUC": 0.5, "PR-AUC": y.mean(),
                   **{f"recall@{c}": c / len(d) for c in (CAPACIDADE, CAPACIDADE_TELEFONE)},
                   **{f"precisao@{c}": y.mean() for c in (CAPACIDADE, CAPACIDADE_TELEFONE)}})
    resultado = pd.DataFrame(linhas).set_index("abordagem")

    _fmt = resultado.copy()
    for col in _fmt.columns:
        if col.startswith(("recall", "precisao")):
            _fmt[col] = _fmt[col].map(_pct)
        elif col.startswith("churns"):
            _fmt[col] = _fmt[col].map(lambda v: "-" if pd.isna(v) else f"{v:.0f}")
        else:
            _fmt[col] = _fmt[col].map(lambda v: f"{v:.3f}")
    print(f"\nA lista de {MES_ALVO}, montada com os dados de {ULTIMO_MES}:")
    print(_fmt.to_string())

    _ref = comparacao.loc["Gradient boosting"]
    _top = aval[aval["na_lista"]]
    print(f"\nChurns esperados pelo modelo entre os {CAPACIDADE}: {_top['risco'].sum():.0f} | "
          f"realizados: {int(_top['churn_realizado'].sum())}")
    print(f"Referencia do teste jan-jun/2026: recall@{CAPACIDADE} {_pct(_ref['recall@120'])}, "
          f"precisao {_pct(_ref['precisao@120'])}")
    _margem = 1.96 * np.sqrt(resultado.iloc[0]['recall@120'] * (1 - resultado.iloc[0]['recall@120'])
                             / y.sum())
    print(f"Margem de erro do recall de um mes so ({y.sum()} churns): +/- {_margem * 100:.0f} pontos")

    # ---- calibracao, clusters e rotas ----------------------------------------------
    _faixa = pd.cut(d["risco"], [0, .05, .2, .5, 1.0001], include_lowest=True,
                    labels=["0-5%", "5-20%", "20-50%", "50-100%"])
    print("\nCalibracao: risco previsto x churn realizado")
    print(d.groupby(_faixa, observed=True)
           .agg(clientes=("churn_realizado", "size"),
                risco_previsto=("risco", lambda s: _pct(s.mean())),
                churn_realizado=("churn_realizado", lambda s: _pct(s.mean())),
                churns=("churn_realizado", "sum"))
           .to_string())

    def _por(col):
        g = d.assign(pego=d["churn_realizado"] * d["na_lista"]).groupby(col)
        t = g.agg(clientes=("churn_realizado", "size"), churns=("churn_realizado", "sum"),
                  na_lista=("na_lista", "sum"), pegos=("pego", "sum"))
        t["recall"] = (t["pegos"] / t["churns"].where(t["churns"] > 0)).map(
            lambda v: "-" if pd.isna(v) else _pct(v))
        return t.sort_values("churns", ascending=False)

    print(f"\nPor cluster:\n{_por('cluster').to_string()}")
    print(f"\nPor rota:\n{_por('Rota_atual').to_string()}")

    # ---- quem a lista errou ---------------------------------------------------------
    _cols = {"risco": "risco", "tpv_total": "TPV no mes escorado",
             "dias_sem_transacao_fim_mes": "dias sem venda fim", "maior_hiato_sem_transacao": "maior hiato",
             "dias_ativos_ultimos_7": "dias ativos ult7", "tempo_ativacao_meses": "meses desde ativ"}
    _pegos = d[(d["churn_realizado"] == 1) & d["na_lista"]]
    _perdidos = d[(d["churn_realizado"] == 1) & ~d["na_lista"]]
    _falsos = d[(d["churn_realizado"] == 0) & d["na_lista"]]
    print("\nPerfil mediano no mes escorado:")
    print(pd.DataFrame({"churns pegos": _pegos[list(_cols)].median(),
                        "churns perdidos": _perdidos[list(_cols)].median(),
                        "alarmes falsos": _falsos[list(_cols)].median()})
            .rename(index=_cols).round(2).to_string())
    _faixa_tel = _perdidos["posicao"].between(CAPACIDADE + 1, CAPACIDADE_TELEFONE)
    print(f"Churns perdidos que estavam na faixa de telefone ({CAPACIDADE + 1}-{CAPACIDADE_TELEFONE}): "
          f"{int(_faixa_tel.sum())} de {len(_perdidos)}")
    _cresceu = _falsos["tpv_total_realizado"] > _falsos["tpv_total"]
    print(f"Alarmes falsos que AUMENTARAM o TPV no mes realizado: {_pct(_cresceu.mean())} "
          f"({int(_cresceu.sum())} de {len(_falsos)})")
    return aval, resultado


ARQUIVO_REALIZADO = arquivo_realizado(MES_ALVO)
if ARQUIVO_REALIZADO is None:
    avaliacao_realizado = resultado_realizado = None
    print(f"Ainda nao ha como validar a lista de {MES_ALVO}: falta data/raw/base_{MES_ALVO}.parquet. "
          f"Gere o arquivo com: python src/pseudonimizar_mes.py \"<carteira bruta do mes>.xlsx\"")
else:
    _realizado, _info = carrega_realizado(ARQUIVO_REALIZADO)
    avaliacao_realizado, resultado_realizado = valida_lista(_realizado, _info)

Carteira de 2026-08: 5,132 linhas
Conferencia do cruzamento: TPV de 2026-07 bate em 100.0% dos 5,065 clientes presentes nos dois meses

Ativos em 2026-07: 1,559 | ausentes do arquivo (indefinidos): 3 | churn em 2026-08: 82 (5.3%)

A lista de 2026-08, montada com os dados de 2026-07:
                                    ROC-AUC PR-AUC churns@120 recall@120 precisao@120 churns@200 recall@200 precisao@200
abordagem                                                                                                               
Gradient boosting (lista entregue)    0.937  0.617         60      73.2%        50.0%         74      90.2%        37.0%
Regra: dias sem venda no fim do mes   0.927  0.457         52      63.4%        43.3%         67      81.7%        33.5%
Sorteio (esperado)                    0.500  0.053          -       7.7%         5.3%          -      12.9%         5.3%

Churns esperados pelo modelo entre os 120: 56 | realizados: 60
Referencia do teste jan-jun/2026: recall@120 62

## 14. Historico de listas (backtest walk-forward)

Reconstroi a lista de cada mes com um modelo treinado so com os meses anteriores. Alimenta a aba *Lista de visitas* do dashboard.

In [14]:
# =============================================================================
# 14. HISTORICO DE LISTAS: BACKTEST WALK-FORWARD
# =============================================================================
# Reconstroi, mes a mes, a lista que teria sido entregue ao time se o modelo ja
# estivesse em operacao. Para cada mes-base M:
#   1. treina o mesmo gradient boosting SO com meses anteriores: linhas com
#      t <= M-1, cujo alvo (TPV de M) ja e conhecido no fechamento de M
#   2. escora os clientes ativos em M e lista os 200 de maior risco para M+1
#   3. compara com o churn realizado em M+1: da propria base ate jun/2026 e da
#      carteira de agosto (celula 13) para a lista de jul/2026
#
# Nenhuma lista enxerga o proprio futuro, por construcao. Sao 13 decisoes
# mensais independentes, cada uma com seu modelo: e o teste mais proximo da
# operacao real. Alimenta a aba "Lista de visitas" do dashboard.

PRIMEIRO_MES_BASE = pd.Period("2025-07", "M")
COLUNAS_LISTA = ["Documento", "Rota_atual", "cluster", "risco", "tpv_total",
                 "tpv_total_medio_m1_m3", "dias_sem_transacao_fim_mes",
                 "maior_hiato_sem_transacao", "dias_ativos_ultimos_7",
                 "tempo_ativacao_meses", "flag_novo_ativo", "flag_reativacao"]

# churn realizado da lista do ultimo mes, vindo da carteira bruta (celula 13)
_realizado_ultimo = None
if avaliacao_realizado is not None:
    _av = avaliacao_realizado.set_index("Documento")
    _realizado_ultimo = _av["churn_realizado"].astype(float).where(_av["presente"])

_inicio = time.perf_counter()
_listas, _metricas = [], []
for mes in pd.period_range(PRIMEIRO_MES_BASE, ULTIMO_MES, freq="M"):
    _treino_m = df_modelo[(df_modelo["conjunto"] == "treino")
                          & (df_modelo["periodo"] <= mes - 1)]
    _ativos_m = df_modelo[df_modelo["periodo"] == mes].copy()

    _modelo_m = boosting().fit(_treino_m[FEATURES], _treino_m["alvo_churn_m1"].astype(int))
    _ativos_m["risco"] = _modelo_m.predict_proba(_ativos_m[FEATURES])[:, 1]
    _ativos_m["score_regra"] = score_regra(_ativos_m)
    _ativos_m["cluster"] = atribui_cluster(_ativos_m)
    _ativos_m["posicao"] = (_ativos_m["risco"].rank(method="first", ascending=False)
                            .astype(int))
    if mes == ULTIMO_MES:
        _ativos_m["churn_realizado"] = (_ativos_m["Documento"].map(_realizado_ultimo)
                                        if _realizado_ultimo is not None else np.nan)
    else:
        _ativos_m["churn_realizado"] = _ativos_m["alvo_churn_m1"].astype(float)

    linha = {"mes_base": str(mes), "mes_lista": str(mes + 1), "ativos": len(_ativos_m),
             "linhas_treino": len(_treino_m),
             "risco_mediano_ativos": float(_ativos_m["risco"].median()),
             "churns_esperados@120": float(_ativos_m.loc[_ativos_m["posicao"] <= CAPACIDADE,
                                                         "risco"].sum())}
    _avaliavel = _ativos_m["churn_realizado"].notna()
    if _avaliavel.any():
        _aval_m = _ativos_m[_avaliavel]
        _y_m = _aval_m["churn_realizado"].astype(int)
        linha["churns_realizados"] = int(_y_m.sum())
        for nome, coluna in (("modelo", "risco"), ("regra", "score_regra")):
            _ordem = _aval_m[coluna].rank(method="first", ascending=False)
            for cap in (CAPACIDADE, CAPACIDADE_TELEFONE):
                pegos = int(_y_m[_ordem <= cap].sum())
                linha[f"{nome}_churns@{cap}"] = pegos
                linha[f"{nome}_recall@{cap}"] = pegos / _y_m.sum() if _y_m.sum() else np.nan
                linha[f"{nome}_precisao@{cap}"] = pegos / int((_ordem <= cap).sum())
    _metricas.append(linha)
    _listas.append(_ativos_m[_ativos_m["posicao"] <= CAPACIDADE_TELEFONE]
                   .assign(mes_base=str(mes), mes_lista=str(mes + 1))
                   [["mes_base", "mes_lista", "posicao"] + COLUNAS_LISTA + ["churn_realizado"]])

historico_listas = (pd.concat(_listas, ignore_index=True)
                      .sort_values(["mes_base", "posicao"]).reset_index(drop=True))
backtest_mensal = pd.DataFrame(_metricas)

# Conferencia: a lista reconstruida do ultimo mes e exatamente a da celula 12
_ultima = historico_listas.loc[historico_listas["mes_base"] == str(ULTIMO_MES), "Documento"]
assert _ultima.tolist() == lista_risco["Documento"].tolist(), \
    "a lista walk-forward do ultimo mes difere da lista entregue na celula 12"

print(f"{len(backtest_mensal)} listas reconstruidas (bases {PRIMEIRO_MES_BASE} a {ULTIMO_MES}), "
      f"{len(backtest_mensal)} modelos treinados em {time.perf_counter() - _inicio:,.0f}s")
print(f"A lista do ultimo mes e identica a entregue na celula 12.\n")

_bt = backtest_mensal.dropna(subset=["modelo_recall@120"])
_tab = pd.DataFrame({
    "lista": _bt["mes_lista"], "ativos": _bt["ativos"], "churns": _bt["churns_realizados"],
    "pegos@120": _bt["modelo_churns@120"],
    "recall@120": (_bt["modelo_recall@120"] * 100).round(1),
    "precisao@120": (_bt["modelo_precisao@120"] * 100).round(1),
    "regra recall@120": (_bt["regra_recall@120"] * 100).round(1),
    "recall@200": (_bt["modelo_recall@200"] * 100).round(1),
    "esperados@120": _bt["churns_esperados@120"].round(0),
})
print("Resultado de cada lista no mes seguinte:")
print(_tab.to_string(index=False))

_churns = _bt["churns_realizados"].sum()
_vence = int((_bt["modelo_churns@120"] > _bt["regra_churns@120"]).sum())
_empata = int((_bt["modelo_churns@120"] == _bt["regra_churns@120"]).sum())
print(f"\nAcumulado de {len(_bt)} listas: {_churns:,} churns")
print(f"  modelo : {_bt['modelo_churns@120'].sum():,} na lista de {CAPACIDADE} "
      f"({_bt['modelo_churns@120'].sum() / _churns:.1%}), precisao "
      f"{_bt['modelo_churns@120'].sum() / (CAPACIDADE * len(_bt)):.1%}; "
      f"lista de {CAPACIDADE_TELEFONE}: {_bt['modelo_churns@200'].sum() / _churns:.1%}")
print(f"  regra  : {_bt['regra_churns@120'].sum():,} na lista de {CAPACIDADE} "
      f"({_bt['regra_churns@120'].sum() / _churns:.1%})")
print(f"  o modelo pegou mais churns que a regra em {_vence} de {len(_bt)} meses "
      f"e empatou em {_empata}")
print(f"  churns esperados x realizados na lista de {CAPACIDADE}: "
      f"{_bt['churns_esperados@120'].sum():,.0f} x {_bt['modelo_churns@120'].sum():,}")

CAMINHO_HISTORICO = caminhos.DADOS_TRATADOS / "listas_risco_historico.parquet"
historico_listas.to_parquet(CAMINHO_HISTORICO, index=False)
print(f"\nGravado: {CAMINHO_HISTORICO.relative_to(caminhos.RAIZ)} "
      f"({len(historico_listas):,} linhas, {CAPACIDADE_TELEFONE} clientes por mes)")

13 listas reconstruidas (bases 2025-07 a 2026-07), 13 modelos treinados em 17s
A lista do ultimo mes e identica a entregue na celula 12.

Resultado de cada lista no mes seguinte:
  lista  ativos  churns  pegos@120  recall@120  precisao@120  regra recall@120  recall@200  esperados@120
2025-08    1507      76         52       68.40         43.30             61.80       86.80          58.00
2025-09    1500      73         54       74.00         45.00             67.10       83.60          57.00
2025-10    1507      72         46       63.90         38.30             55.60       86.10          52.00
2025-11    1523      87         51       58.60         42.50             47.10       74.70          54.00
2025-12    1511      87         57       65.50         47.50             59.80       83.90          53.00
2026-01    1515      76         51       67.10         42.50             59.20       85.50          63.00
2026-02    1524     128         73       57.00         60.80             52.30 

## Conclusoes (execucao de 12/09/2026)

1. **Com 120 abordagens por mes, o gradient boosting pos na lista 62,6% dos churns do mes
   seguinte**, com precisao de 48,8%: quase metade da lista de fato sairia. Uma lista sorteada
   pegaria 8%.
2. **A regra simples ja resolve a maior parte.** Ranquear por dias sem venda no fim do mes pega
   59,5%; o modelo acrescenta cerca de 3 churns por mes na lista, e a regressao logistica empata
   com o boosting (62,4%). O sinal dominante e o comportamento diario do proprio mes.
3. **Com 200 abordagens (telefone), o recall sobe para 79%**, com precisao de 37%.
4. **Qtd_equipamentos ficou fora por suspeita de vazamento** (celula 3). Com ela, o recall@120
   seria 64,7%. Se o dono do dado confirmar que a contagem e do ultimo dia do mes, pode voltar.
5. **Onde o modelo nao chega:** churn de cliente maduro que vendia normalmente (C5, recall de 45%).
   Esse churn nao deixa rastro no mes anterior; prever com mais antecedencia e a proxima frente.
6. **13 features nao superam o proprio ruido** na importancia por permutacao (celula 10), entre
   elas mes_calendario, tempo_ativacao_meses e as categoricas de canal, documento e RAV. Sao as
   candidatas a sair numa proxima rodada.
7. **Lista de ago/2026** gravada em `data/processed/lista_risco_2026-08.parquet`, com 200 clientes.
   Dos 120 primeiros, 88 sao do C2; por rota, Parnaiba Sul (34), Parnaiba Centro (24),
   Ceara Oeste (20) e Ceara Leste (21) concentram 83%.
8. **Validacao real em ago/2026** (celula 13). Dos 82 clientes que churnaram em agosto, **60
   estavam na lista de 120**: recall de 73,2% e precisao de 50,0%, acima do teste de jan-jun.
   O modelo esperava ~56 churns na lista. A regra simples teria pego 52; a lista de 200 pegaria
   90,2%. Os 22 perdidos pareciam saudaveis em julho (risco mediano de 8%), e 14 deles estavam
   entre as posicoes 121 e 200. E um mes so: margem de cerca de 10 pontos no recall.
9. **Daqui em diante, grupo de controle.** A validacao de agosto foi limpa porque ninguem agiu
   sobre a lista. Quando o time abordar os clientes, os retidos vao parecer alarmes falsos;
   sorteie parte da lista para nao abordar e meca o acerto nesse grupo.
10. **Backtest walk-forward** (celula 14). Reconstruindo 13 listas mensais (ago/2025 a ago/2026),
    cada uma com um modelo treinado so com o passado, **726 dos 1.114 churns estavam na lista de
    120 (65,2%)**, com precisao de 46,5%; a lista de 200 pegaria 83,4%. A regra simples pegou 660
    (59,2%), e o modelo a superou em 11 dos 13 meses. Churns esperados na soma das probabilidades:
    756, contra 726 realizados. As listas alimentam a aba *Lista de visitas* do dashboard.